In [1]:
import geopandas as gpd

In [3]:
tiles_locs = gpd.read_file("utils/resources/tiles_locs/ch.swisstopo.images-swissimage-dop10.metadata.shp")

ids = tiles_locs.id.values
E = [x.split('_')[0] for x in ids]
N = [x.split('_')[1] for x in ids]
EN = [[int(x), int(y)] for x,y in zip(E,N)]

In [4]:

cantons = gpd.read_file('utils/resources/swissboundaries/swissBOUNDARIES3D_1_5_TLM_KANTONSGEBIET.shp')

canton_polygons = cantons[cantons.NAME == "Vaud"]
Emin = int(canton_polygons.bounds.minx.values[0] // 1000)
Emax = int((canton_polygons.bounds.maxx.values[0] + 1) // 1000)
Nmin = int(canton_polygons.bounds.miny.values[0] // 1000)
Nmax = int((canton_polygons.bounds.maxy.values[0] + 1) // 1000)

tiles_to_download = [x for x in EN if Emin <= x[0] <= Emax and Nmin <= x[1] <= Nmax]

In [5]:
print(len(tiles_to_download))

5945


In [9]:
print(tiles_to_download)

[[2532, 1154], [2563, 1199], [2505, 1143], [2499, 1152], [2566, 1199], [2556, 1164], [2544, 1184], [2504, 1124], [2532, 1177], [2581, 1200], [2578, 1154], [2554, 1152], [2573, 1201], [2523, 1152], [2546, 1150], [2578, 1121], [2525, 1179], [2561, 1130], [2509, 1158], [2542, 1180], [2571, 1195], [2534, 1160], [2544, 1151], [2538, 1153], [2568, 1170], [2526, 1161], [2563, 1174], [2502, 1116], [2512, 1163], [2506, 1134], [2502, 1158], [2509, 1135], [2558, 1201], [2569, 1155], [2585, 1131], [2531, 1166], [2584, 1191], [2518, 1157], [2521, 1155], [2566, 1162], [2569, 1165], [2538, 1155], [2545, 1148], [2566, 1198], [2576, 1197], [2539, 1149], [2556, 1152], [2499, 1130], [2506, 1123], [2570, 1124], [2540, 1196], [2578, 1136], [2532, 1167], [2554, 1120], [2506, 1144], [2527, 1175], [2581, 1130], [2570, 1166], [2568, 1200], [2581, 1119], [2525, 1144], [2500, 1138], [2575, 1161], [2564, 1148], [2564, 1124], [2515, 1172], [2528, 1189], [2552, 1144], [2532, 1162], [2583, 1150], [2509, 1161], [2557

In [8]:
print(cantons.bounds)

           minx         miny         maxx         maxy
0   2485410.215  1109644.837  2512974.023  1135578.430
1   2692317.346  1248320.476  2755679.566  1283503.921
2   2548579.410  1078560.231  2679786.815  1167428.364
3   2620698.052  1221173.284  2676826.750  1274772.439
4   2672167.282  1193656.909  2718719.376  1231066.700
5   2669244.906  1223895.629  2716900.400  1283342.861
6   2646044.797  1178345.137  2681479.151  1203582.462
7   2546837.021  1143053.265  2595530.560  1206219.718
8   2709273.640  1183890.430  2738138.460  1225966.591
9   2673344.851  1153451.251  2715795.707  1205338.784
10  2659303.794  1180482.335  2686449.010  1208074.866
11  2592560.720  1213703.190  2644759.746  1261330.177
12  2732375.301  1234614.715  2765346.399  1259711.918
13  2554671.133  1222309.156  2609050.776  1261608.177
14  2692837.746  1114568.595  2833857.724  1214817.191
15  2494306.344  1115148.823  2585462.064  1204075.563
16  2630127.986  1180568.481  2681764.349  1237690.711
17  267248

In [11]:
# make sure CRS match
if tiles_locs.crs != canton_polygons.crs:
    tiles_locs = tiles_locs.to_crs(canton_polygons.crs)
# spatial join - keep tiles that intersect Vaud
tiles_vaud = gpd.sjoin(tiles_locs, canton_polygons, how="inner", predicate="intersects")

# sjoin adds columns from canton_polygons (suffixed if names clash) - 
# drop them if you just want the original tile columns back
tiles_vaud = tiles_vaud[tiles_locs.columns]

In [ ]:
print(len(tiles_vaud))
print(tiles_vaud)
Emin = int(tiles_vaud.bounds.minx.values[0] // 1000)
Emax = int((tiles_vaud.bounds.maxx.values[0] + 1) // 1000)
Nmin = int(tiles_vaud.bounds.miny.values[0] // 1000)
Nmax = int((tiles_vaud.bounds.maxy.values[0] + 1) // 1000)

tiles_to_download_vaud = [x for x in EN if Emin <= x[0] <= Emax and Nmin <= x[1] <= Nmax]
tiles_to_download_vaud = [[int(x), int(y)] for loc in tiles_vaud.id.to_list() for [x,y] in loc.split('_')]
print(tiles_to_download_vaud)
print(len(tiles_to_download_vaud))

3553
              id datenstand resolution  \
3163   2532_1154       2023      10 cm   
3172   2563_1199       2023      10 cm   
3185   2505_1143       2023      10 cm   
3189   2499_1152       2023      10 cm   
3190   2566_1199       2023      10 cm   
...          ...        ...        ...   
37953  2558_1176       2023      10 cm   
37959  2568_1120       2023      10 cm   
37961  2579_1141       2023      25 cm   
37964  2502_1145       2023      10 cm   
37970  2565_1122       2023      10 cm   

                                                geometry  
3163   POLYGON ((2532000 1154000, 2532000 1155000, 25...  
3172   POLYGON ((2563000 1199000, 2563000 1200000, 25...  
3185   POLYGON ((2505000 1143000, 2505000 1144000, 25...  
3189   POLYGON ((2499000 1152000, 2499000 1153000, 25...  
3190   POLYGON ((2566000 1199000, 2566000 1200000, 25...  
...                                                  ...  
37953  POLYGON ((2558000 1176000, 2558000 1177000, 25...  
37959  POLYGON ((2

ValueError: too many values to unpack (expected 2)

In [ ]:
print(tiles_vaud.id.to_list())
x,y = tiles_vaud.id.to_list()[0].split('_')
print(int(x), int(y))
print([x.split('_') for x in tiles_vaud.id.to_list()])
print([[int(x) for x in tile.split('_')] for tile in tiles_vaud.id.to_list()])
print([[int(x) for x in tile.split('_')] for tile in ])
# print([a for x in tiles_vaud.id.to_list() for a in x.split('_')])

['2532_1154', '2563_1199', '2505_1143', '2499_1152', '2566_1199', '2544_1184', '2532_1177', '2523_1152', '2546_1150', '2525_1179', '2561_1130', '2509_1158', '2542_1180', '2571_1195', '2534_1160', '2544_1151', '2538_1153', '2526_1161', '2512_1163', '2506_1134', '2502_1158', '2509_1135', '2531_1166', '2518_1157', '2521_1155', '2538_1155', '2545_1148', '2566_1198', '2539_1149', '2499_1130', '2570_1124', '2540_1196', '2578_1136', '2532_1167', '2506_1144', '2527_1175', '2581_1130', '2568_1200', '2525_1144', '2500_1138', '2564_1148', '2564_1124', '2515_1172', '2528_1189', '2552_1144', '2532_1162', '2583_1150', '2509_1161', '2557_1148', '2514_1148', '2509_1144', '2538_1162', '2577_1133', '2522_1162', '2536_1190', '2545_1176', '2560_1147', '2575_1149', '2530_1146', '2538_1157', '2579_1125', '2531_1169', '2551_1181', '2537_1164', '2526_1147', '2501_1127', '2564_1186', '2559_1195', '2536_1146', '2528_1163', '2539_1168', '2562_1143', '2542_1190', '2581_1150', '2533_1150', '2549_1157', '2527_1147'